##NBA Multi-Dimensional Analytics & Scouting Engine

A data pipeline and analytics framework for integrating NBA tracking, synergy, box score, and shot data into a unified player-level dataset.

**Key Features**

Multi-source data integration (tracking, synergy, box score, shot locations)
Centralized master dataset with 500+ engineered features
Regular season vs playoff comparison for contextual analysis
Modular ingestion pipeline with caching (Parquet-based)
Built for downstream modeling (e.g., RAPM, salary prediction)

# Import Packages

In [85]:
# Standard library
import gc
import itertools
import pickle
import time
import warnings
from functools import reduce
from random import sample
from unidecode import unidecode
import re, requests
from pathlib import Path
import sklearn
import shap
import joblib
import json

# Core scientific stack
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from matplotlib.patches import Rectangle
from matplotlib.ticker import FormatStrFormatter
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_absolute_error

import holoviews as hv
import hvplot.pandas  # noqa: F401 (registers hvplot accessor)
hv.extension("bokeh")

# Stats
from scipy import stats

# NBA API
from nba_api.stats.static.teams import get_teams
from nba_api.stats.static.players import get_players, get_active_players
from nba_api.stats.endpoints import leaguegamefinder, leaguedashplayerstats

import importlib
import nba_viz_utils
importlib.reload(nba_viz_utils)


from nba_viz_utils import *

In [2]:
pwd

'/Users/siddharthravindran/Documents/nba_data_vis'

# Upload Master

## Load Master Data Frame

In [3]:
df_master = pd.read_parquet(
  "data/df_master_current.parquet"
).set_index(INDEX_COLS).sort_index()

## Master Integrity Check

In [4]:
check_master_integrity(df_master)

📊 Integrity Check for df_master:
  - Shape: (8373, 730)
  - Total Players: 1556
  - Seasons Present: ['2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
  - ✅ No duplicate indices found.
  - ✅ All columns contain at least some data.


# Missing Data Check

In [4]:
nan_by_season = df_master.groupby(level='SEASON').apply(lambda g: g.isna().mean())
flip = nan_by_season.columns[(nan_by_season.max() > 0.95) & (nan_by_season.min() < 0.5)]
nan_by_season[flip].round(2)

""
SEASON
2015-16
2016-17
2017-18
2018-19
2019-20
2020-21
2021-22
2022-23
2023-24


# Transpose Regular Season & Playoffs

In [5]:
model = build_model_table(df_master)

In [7]:
m = model.reset_index()
m['PLAYER_NAME'] = m['PLAYER_NAME'].apply(norm_name)
print("dupe player-seasons:", m.duplicated(subset=['PLAYER_ID','SEASON']).sum())

dupe player-seasons: 0


In [3]:
# m.to_parquet("data/transposed_df_master.parquet")

model = pd.read_parquet("data/transposed_df_master.parquet")

In [4]:
model

,index,PLAYER_ID,SEASON,PLAYER_NAME,TEAM_ABBREVIATION,TEAM_ID,GP_rs,W_rs,L_rs,W_PCT_rs,...,CLUTCH_TS_PCT_po,CLUTCH_USG_PCT_po,CLUTCH_E_PACE_po,CLUTCH_PACE_po,CLUTCH_PACE_PER40_po,CLUTCH_PIE_po,CLUTCH_POSS_po,CLUTCH_FGM_PG_po,CLUTCH_FGA_PG_po,made_playoffs
0,0,201950,2019-20,jrue holiday,NOP,1610612740,61,26,35,0.426,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,1,1627759,2019-20,jaylen brown,BOS,1610612738,57,38,19,0.667,...,0.444,0.186,94.69,94.03,78.36,0.118,100.0,0.5,1.5,1
2,2,203935,2019-20,marcus smart,BOS,1610612738,60,39,21,0.650,...,0.429,0.116,93.94,92.74,77.28,0.063,96.0,0.2,0.6,1
3,3,1628970,2019-20,miles bridges,CHA,1610612766,65,23,42,0.354,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,4,202710,2019-20,jimmy butler,MIA,1610612748,58,38,20,0.655,...,0.760,0.342,88.07,87.53,72.94,0.341,96.0,0.9,1.5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5963,5963,204066,2018-19,john holland,CLE,1610612739,1,0,1,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
5964,5964,1629122,2019-20,jp macura,CLE,1610612739,1,1,0,1.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
5965,5965,1630701,2022-23,michael foster,PHI,1610612755,1,1,0,1.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
5966,5966,1628382,2023-24,justin jackson,MIN,1610612750,2,2,0,1.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


# Salaries

In [ ]:
all_urls = harvest_player_urls()        # all 11 seasons → ~1,800 unique players
salaries  = scrape_all(all_urls)
print(salaries.shape); salaries.head()

  2015-16: 487 players (total unique 487)
  2016-17: 494 players (total unique 586)
  2017-18: 549 players (total unique 722)
  2018-19: 537 players (total unique 830)
  2019-20: 538 players (total unique 949)
  2020-21: 547 players (total unique 1043)
  2021-22: 613 players (total unique 1170)
  2022-23: 548 players (total unique 1255)
  2023-24: 578 players (total unique 1355)
  2024-25: 575 players (total unique 1459)
  2025-26: 588 players (total unique 1562)
  50/1562
  100/1562
  150/1562
  200/1562
  250/1562
  300/1562
  350/1562
  400/1562
  450/1562
  500/1562
  550/1562
  600/1562
  650/1562
  700/1562
  750/1562
  800/1562
  850/1562
  900/1562
  950/1562
  1000/1562
  1050/1562
  1100/1562
  1150/1562
  1200/1562
  1250/1562
  1300/1562
  1350/1562
  1400/1562
  1450/1562
  1500/1562
  1550/1562
(7701, 4)


,SEASON,Salary,Player,name_key
0,2016-17,5994764,Álex Abrines,alex abrines
1,2017-18,5725000,Álex Abrines,alex abrines
2,2018-19,3575183,Álex Abrines,alex abrines
3,2020-21,2582160,Precious Achiuwa,precious achiuwa
4,2021-22,2711280,Precious Achiuwa,precious achiuwa


In [6]:
# salaries.to_csv(DATA_DIR / "bbref_salaries_2015-2025.csv", index=False)

salaries = pd.read_csv("data/bbref_salaries_2015-2025.csv")
salaries = salaries[salaries['SEASON'].between('2015-16', '2024-25')].reset_index(drop=True)

In [7]:
salaries_25_26 = pd.read_csv("data/contracts.csv")

In [ ]:
col_2526 = next(c for c in salaries_25_26.columns if '2025' in str(c))   # column NAME → separate var

def to_int_salary(s):
    digits = re.sub(r'[^0-9]', '', str(s).split('.')[0])   # split('.') guards against a float '45780966.0'
    return int(digits) if digits else 0

current = salaries_25_26[['Player', col_2526]].dropna().copy()
current['Salary']   = current[col_2526].map(to_int_salary)
current = current[current['Salary'] > 0]
current['SEASON']   = '2025-26'
current['name_key'] = current['Player'].apply(norm_name)
current = current[['SEASON', 'Salary', 'Player', 'name_key']]
print(current.shape); current.head()

(529, 4)


,SEASON,Salary,Player,name_key
0,2025-26,59606817,Stephen Curry,stephen curry
1,2025-26,55224526,Joel Embiid,joel embiid
2,2025-26,55224526,Nikola Jokić,nikola jokic
3,2025-26,54708609,Kevin Durant,kevin durant
4,2025-26,54126450,Jayson Tatum,jayson tatum


In [32]:
salaries = pd.concat([salaries, current[['SEASON', 'Salary', 'Player', 'name_key']]], ignore_index=True)
salaries = salaries.drop_duplicates(['name_key', 'SEASON'])

In [4]:
# salaries.to_csv("data/bbref_salaries_2015-2026.csv", index=False)
salaries = pd.read_csv("data/bbref_salaries_2015-2026.csv")

# Build Model Data Frame

In [5]:
model = attach_salaries(model, salaries)    

In [6]:
# did the scrape actually fix the ~14% hole?
real = model[(model['GP_rs'] >= 20) & (model['MIN_rs'] >= 10)]
print(f"coverage (real-minute players): {real['Salary'].notna().mean():.1%}  "
      f"| missing rows: {real['Salary'].isna().sum()}")

coverage (real-minute players): 97.7%  | missing rows: 100


## Experience

In [7]:
# draft / experience — model-layer attributes, merged on PLAYER_ID
model['PLAYER_ID'] = model['PLAYER_ID'].astype(str)
bio = get_draft_table(SEASONS)
bio['PLAYER_ID'] = bio['PLAYER_ID'].astype(str)
model = (model.drop(columns=['DRAFT_POSITION', 'DRAFT_YR'], errors='ignore')   # idempotent
              .merge(bio, on='PLAYER_ID', how='left'))

model['IS_UNDRAFTED']     = model['DRAFT_POSITION'].isna().astype(int)
model['EXPERIENCE']       = (model['SEASON'].str[:4].astype(int) - model['DRAFT_YR']).clip(lower=0)
model['MAX_PCT_ELIGIBLE'] = model['EXPERIENCE'].map(
    lambda e: np.nan if pd.isna(e) else (0.25 if e <= 6 else 0.30 if e <= 9 else 0.35))

# target + filter LAST (so `model` stays the full scoreable population)
model['pct_cap'] = model['Salary'] / model['SEASON'].map(SALARY_CAP)

## All-NBA

In [8]:
allnba_raw = fetch_all_nba_raw()
allnba_raw.head(12)

status: 200
shape: (304, 9) | cols: ['Season', 'Lg', 'Tm', 'Voting', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8']


,Season,Lg,Tm,Voting,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,2025-26,NBA,1st,(V),Nikola Jokić C,Victor Wembanyama C,Shai Gilgeous-Alexander G,Luka Dončić G,Cade Cunningham G
1,2025-26,NBA,2nd,(V),Jaylen Brown F,Kawhi Leonard F,Kevin Durant F,Donovan Mitchell G,Jalen Brunson G
2,2025-26,NBA,3rd,(V),Jalen Duren C,Jalen Johnson F,Chet Holmgren F,Tyrese Maxey G,Jamal Murray G
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-25,NBA,1st,(V),Nikola Jokić C,Giannis Antetokounmpo F,Jayson Tatum F,Shai Gilgeous-Alexander G,Donovan Mitchell G
5,2024-25,NBA,2nd,(V),Evan Mobley C,LeBron James F,Stephen Curry G,Anthony Edwards G,Jalen Brunson G
6,2024-25,NBA,3rd,(V),Karl-Anthony Towns C,Jalen Williams F,Cade Cunningham G,Tyrese Haliburton G,James Harden G
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2023-24,NBA,1st,(V),Nikola Jokić C,Giannis Antetokounmpo F,Jayson Tatum F,Luka Dončić G,Shai Gilgeous-Alexander G
9,2023-24,NBA,2nd,(V),Anthony Davis C,Kevin Durant F,Kawhi Leonard F,Jalen Brunson G,Anthony Edwards G


In [10]:
def prior_counts(name_key, year, decay=0.75):
    p = picks[picks['name_key'] == name_key]
    last3 = int(p[(p['yr'] < year) & (p['yr'] >= year - 3)]['n'].sum())
    past  = p[p['yr'] < year]
    # decayed career count: each past selection weighted by decay^(years_ago)
    ever  = float((past['n'] * (decay ** (year - past['yr']))).sum())
    return last3, ever

def prior_weighted(name_key, year, decay=0.75):
    p = aw[aw['name_key'] == name_key]
    last3 = int(p[(p['yr'] < year) & (p['yr'] >= year - 3)]['weight'].sum())
    past  = p[p['yr'] < year]
    ever  = float((past['weight'] * (decay ** (year - past['yr']))).sum())
    return ever, last3

In [11]:
# --- parse the raw All-NBA table → long (name_key, SEASON, team_level) ---
player_cols = ['Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8']

an = allnba_raw[allnba_raw['Lg'] == 'NBA'].copy()          # drops blank rows + any ABA history
an = an.melt(id_vars=['Season', 'Tm'], value_vars=player_cols, value_name='player_raw')
an = an.dropna(subset=['player_raw'])

# strip the trailing position token(s): "Giannis Antetokounmpo F" / "... G-F" → name
an['player'] = an['player_raw'].str.replace(r'\s+[CFG](-[CFG])?$', '', regex=True)
an['name_key'] = an['player'].apply(norm_name)
an = an.rename(columns={'Season': 'SEASON'})
an = an[an['SEASON'].isin(SEASONS)][['name_key', 'SEASON', 'Tm']]
print("All-NBA selections parsed:", an.shape)
print(an.head())

# --- credential features: All-NBA in PRIOR seasons (causally clean + matches supermax rule) ---
an['yr'] = an['SEASON'].str[:4].astype(int)
picks = an.groupby(['name_key', 'yr']).size().rename('n').reset_index()   # 1 row per player-season selected

model['name_key'] = model['PLAYER_NAME'].apply(norm_name)
yrs = model['SEASON'].str[:4].astype(int)
counts = [prior_counts(nk, y) for nk, y in zip(model['name_key'], yrs)]
model['ALLNBA_PRIOR3']     = [c[0] for c in counts]      # selections in trailing 3 seasons
model['ALLNBA_PRIOR_EVER'] = [c[1] for c in counts]      # career selections before this season
model = model.drop(columns='name_key')

print("\nplayers with ALLNBA_PRIOR3 > 0:", (model['ALLNBA_PRIOR3'] > 0).sum())
print(model[model['ALLNBA_PRIOR3'] > 0][['PLAYER_NAME','SEASON','ALLNBA_PRIOR3','ALLNBA_PRIOR_EVER','pct_cap']]
      .sort_values('ALLNBA_PRIOR_EVER', ascending=False).head(10))

All-NBA selections parsed: (165, 3)
       name_key   SEASON   Tm
0  nikola jokic  2025-26  1st
1  jaylen brown  2025-26  2nd
2   jalen duren  2025-26  3rd
3  nikola jokic  2024-25  1st
4   evan mobley  2024-25  2nd

players with ALLNBA_PRIOR3 > 0: 243
                PLAYER_NAME   SEASON  ALLNBA_PRIOR3  ALLNBA_PRIOR_EVER  \
140            lebron james  2025-26              3           2.831059   
62             lebron james  2024-25              3           2.774746   
1174  giannis antetokounmpo  2025-26              3           2.774746   
628   giannis antetokounmpo  2024-25              3           2.699661   
260            lebron james  2023-24              3           2.699661   
756           stephen curry  2025-26              3           2.653081   
371            nikola jokic  2025-26              3           2.599548   
1203  giannis antetokounmpo  2023-24              3           2.599548   
83             lebron james  2022-23              3           2.599548   
361    

In [12]:
# weight selection quality: 1st = 3, 2nd = 2, 3rd = 1
TM_WEIGHT = {'1st': 3, '2nd': 2, '3rd': 1}
an['weight'] = an['Tm'].map(TM_WEIGHT)

aw = an.dropna(subset=['weight']).copy()
aw['yr'] = aw['SEASON'].str[:4].astype(int)

# # weighted credential: prior-ever and prior-3 in WEIGHTED points, not raw counts
# def prior_weighted(name_key, year):
#     p = aw[aw['name_key'] == name_key]
#     ever  = int(p[p['yr'] <  year]['weight'].sum())
#     last3 = int(p[(p['yr'] < year) & (p['yr'] >= year - 3)]['weight'].sum())
#     return ever, last3

model['name_key'] = model['PLAYER_NAME'].apply(norm_name)
yrs = model['SEASON'].str[:4].astype(int)
wc = [prior_weighted(nk, y) for nk, y in zip(model['name_key'], yrs)]
model['ALLNBA_WT_EVER'] = [c[0] for c in wc]
model['ALLNBA_WT3']     = [c[1] for c in wc]
model = model.drop(columns='name_key')

# check the separation you care about
print(model[model['PLAYER_NAME'].str.contains('giannis|embiid', case=False, na=False)]
      [['PLAYER_NAME','SEASON','ALLNBA_WT_EVER','ALLNBA_WT3']]
      .query("SEASON == '2025-26'").to_string(index=False))

          PLAYER_NAME  SEASON  ALLNBA_WT_EVER  ALLNBA_WT3
          joel embiid 2025-26         2.84024           3
giannis antetokounmpo 2025-26         8.14904           9


## Availability

In [14]:
# availability — explicit, so the model stops confusing "hurt" with "declined"
model['GP_rs']  = model['GP_rs'].fillna(0)
model['AVAILABILITY'] = model['GP_rs'] / 82.0                 # share of season played
model['TOTAL_MIN_rs'] = model['MIN_rs'] * model['GP_rs']      # volume: high only if good AND available

## Lagged Features

In [15]:
# the spine to lag — production identity, not all 1000 cols
SPINE = ['MIN_rs','FGM_rs','FTM_rs','PIE_rs','USG_PCT_rs','FRONT_CT_TOUCHES_rs',
         'POST_TOUCHES_rs','CLOSESTDEF_4_6_FGM_rs','GP_rs','TOTAL_MIN_rs',
         'ALLNBA_WT_EVER']
SPINE = [c for c in SPINE if c in model.columns]

# integer season key so shift respects chronological order
model['_yr'] = model['SEASON'].str[:4].astype(int)
model = model.sort_values(['PLAYER_ID','_yr'])

# lag-1 and lag-2: the player's prior-season and two-seasons-ago production
for k in (1, 2):
    lagged = model.groupby('PLAYER_ID')[SPINE].shift(k)
    # only valid if the prior row is the IMMEDIATELY preceding season (no gap)
    prev_yr = model.groupby('PLAYER_ID')['_yr'].shift(k)
    gap_ok  = (model['_yr'] - prev_yr) == k
    lagged  = lagged.where(gap_ok)                      # NaN out skipped/missing seasons
    lagged.columns = [f"{c}_lag{k}" for c in SPINE]
    model = pd.concat([model, lagged], axis=1)

model = model.drop(columns='_yr')

# sanity: Giannis should now carry last year's healthy minutes alongside this year's
print(model[model['PLAYER_NAME'].str.contains('giannis antetokounmpo', case=False, na=False)]
      [['SEASON','MIN_rs','MIN_rs_lag1','GP_rs','GP_rs_lag1']].sort_values('SEASON').to_string(index=False))

 SEASON  MIN_rs  MIN_rs_lag1  GP_rs  GP_rs_lag1
2015-16    35.3          NaN     80         NaN
2016-17    35.6         35.3     80        80.0
2017-18    36.7         35.6     75        80.0
2018-19    32.8         36.7     72        75.0
2019-20    30.4         32.8     63        72.0
2020-21    33.0         30.4     61        63.0
2021-22    32.9         33.0     67        61.0
2022-23    32.1         32.9     63        67.0
2023-24    35.2         32.1     73        63.0
2024-25    34.2         35.2     67        73.0
2025-26    28.9         34.2     36        67.0


## LEBRON

In [8]:
pwd

'/Users/siddharthravindran/Documents/nba_data_vis'

In [6]:
lebron = load_lebron_seasons("data")  # stacks all seasons present
mdf = merge_lebron(mdf, lebron) 

✅ Loaded LEBRON for seasons: ['2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']
   4481 player-season rows, cols: ['LBR_LEBRON', 'LBR_O_LEBRON', 'LBR_D_LEBRON', 'LBR_WAR', 'LBR_OffRole', 'LBR_DefRole']
🔗 LEBRON matched 4223/4228 model rows (100%). Unmatched are pre-2016 or sub-threshold players.


## Finalize Model Data Frame

In [16]:
model = model.loc[:, ~model.columns.duplicated()]

mdf = model[model['Salary'].notna() & (model['GP_rs'] >= 20) & (model['MIN_rs'] >= 10)].copy()
mdf['pct_cap'] = mdf['Salary'] / mdf['SEASON'].map(SALARY_CAP)
mdf = mdf[mdf['pct_cap'].notna()].copy()

# does the credential separate the pay tiers the way we expect?
print(mdf.groupby(mdf['ALLNBA_PRIOR3'].clip(upper=3))['pct_cap'].agg(['mean','count']))
print("mdf:", mdf.shape)
print(mdf['pct_cap'].describe())

                   mean  count
ALLNBA_PRIOR3                 
0              0.070139   3994
1              0.251553    130
2              0.285717     64
3              0.330198     40
mdf: (4228, 1501)
count    4228.000000
mean        0.081441
std         0.084130
min         0.000047
25%         0.018757
50%         0.046164
75%         0.116588
max         0.407253
Name: pct_cap, dtype: float64


In [11]:
mdf = lag_and_encode_lebron(mdf)   # mdf already has the merged LBR_* columns

✅ Built 30 lagged numeric features + 22 lagged role one-hots.
   Dropped contemporaneous LBR_* columns (leak-safe).


In [20]:
[c for c in mdf.columns if 'D_LEBRON' in c or 'Role_prior' in c]

['LBR_D_LEBRON_prior1',
 'LBR_D_LEBRON_prior2',
 'LBR_OffRole_prior1_Athletic Finisher',
 'LBR_OffRole_prior1_Low Minute',
 'LBR_OffRole_prior1_Movement Shooter',
 'LBR_OffRole_prior1_Off Screen Shooter',
 'LBR_OffRole_prior1_Post Scorer',
 'LBR_OffRole_prior1_Primary Ball Handler',
 'LBR_OffRole_prior1_Roll + Cut Big',
 'LBR_OffRole_prior1_Secondary Ball Handler',
 'LBR_OffRole_prior1_Shot Creator',
 'LBR_OffRole_prior1_Slasher',
 'LBR_OffRole_prior1_Stationary Shooter',
 'LBR_OffRole_prior1_Stretch Big',
 'LBR_OffRole_prior1_Versatile Big',
 'LBR_OffRole_prior1_nan',
 'LBR_DefRole_prior1_Anchor Big',
 'LBR_DefRole_prior1_Chaser',
 'LBR_DefRole_prior1_Helper',
 'LBR_DefRole_prior1_Low Activity',
 'LBR_DefRole_prior1_Mobile Big',
 'LBR_DefRole_prior1_Point of Attack',
 'LBR_DefRole_prior1_Wing Stopper',
 'LBR_DefRole_prior1_nan']

### Check All-NBA Weight

In [19]:
S = "2025-26"   # a recent season
snap = mdf[mdf["SEASON"] == S]

# faded stars: made All-NBA at some point, but nothing in the last 3 years
faded = snap[(snap["ALLNBA_PRIOR_EVER"] > 0) & (snap["ALLNBA_PRIOR3"] == 0)] \
          .sort_values("ALLNBA_WT_EVER", ascending=False)

print(f"=== Faded stars in {S}: career All-NBA, none in last 3 yrs ===")
print(faded[["PLAYER_NAME","ALLNBA_PRIOR_EVER","ALLNBA_WT_EVER","ALLNBA_PRIOR3","pct_cap"]]
      .head(15).to_string(index=False))

print("\n=== Current stars (recent All-NBA) — decay should barely touch these ===")
current = snap[snap["ALLNBA_PRIOR3"] > 0].sort_values("ALLNBA_WT_EVER", ascending=False)
print(current[["PLAYER_NAME","ALLNBA_PRIOR_EVER","ALLNBA_WT_EVER","ALLNBA_PRIOR3","ALLNBA_WT3"]]
      .head(10).to_string(index=False))

=== Faded stars in 2025-26: career All-NBA, none in last 3 yrs ===
      PLAYER_NAME  ALLNBA_PRIOR_EVER  ALLNBA_WT_EVER  ALLNBA_PRIOR3  pct_cap
    demar derozan           0.491604        0.908123              0 0.158878
russell westbrook           0.542974        0.905883              0 0.014848
      paul george           0.527215        0.794183              0 0.334090
      rudy gobert           0.623852        0.698936              0 0.226322
    pascal siakam           0.494385        0.672363              0 0.294545
        ja morant           0.316406        0.632812              0 0.255072
   draymond green           0.131398        0.187712              0 0.167432
   andre drummond           0.056314        0.056314              0 0.032332
    klay thompson           0.056314        0.056314              0 0.107772

=== Current stars (recent All-NBA) — decay should barely touch these ===
            PLAYER_NAME  ALLNBA_PRIOR_EVER  ALLNBA_WT_EVER  ALLNBA_PRIOR3  ALLNBA_WT3
  g

### Load/Save Model Data Frame

In [6]:
# mdf.to_parquet("data/transposed_df_master_with_bio.parquet")
mdf = pd.read_parquet("data/transposed_df_master_with_bio.parquet")

In [3]:
# mdf.to_parquet('data/transposed_df_master_with_bio_v2.parquet')
# print("saved transposed_df_master_with_bio_v2:", mdf.shape)

mdf = pd.read_parquet("data/transposed_df_master_with_bio_v2.parquet")

In [4]:
# what did the salary merge join ON? confirm those keys exist and look right
print("columns that look like join keys:")
print([c for c in mdf.columns if c.upper() in ('PLAYER_NAME','SEASON','PLAYER_ID','NBA_ID','YEAR')])
print("\nsample of those key columns:")
key_cols = [c for c in ['PLAYER_NAME','SEASON','PLAYER_ID','nba_id','YEAR'] if c in mdf.columns]
print(mdf[key_cols].head())

columns that look like join keys:
['PLAYER_ID', 'SEASON', 'PLAYER_NAME']

sample of those key columns:
          PLAYER_NAME   SEASON PLAYER_ID
3950     andrew bogut  2015-16    101106
5017     andrew bogut  2016-17    101106
2971  marvin williams  2015-16    101107
2152  marvin williams  2016-17    101107
2501  marvin williams  2017-18    101107


# Train Model

In [12]:

# 4) re-prefilter (regenerates keep WITH the new cols) then retrain
keep, report = prefilter_features(mdf); print(report)
feats = keep + [c for c in ['AVAILABILITY','TOTAL_MIN_rs'] if c not in keep]
print("availability in feats?", 'AVAILABILITY' in feats, '| total_min in feats?', 'TOTAL_MIN_rs' in feats)

feats.remove('DRAFT_YR')   # not a real feature, just a proxy for experience
print("draft year in feats?", 'DRAFT_YR' in feats)

{'start': 1495, 'const': 1, 'empty': 2, 'collinear': 449, 'keep': 1043}
availability in feats? True | total_min in feats? True
draft year in feats? False


In [13]:
train = mdf[mdf['SEASON'] <= '2021-22']
valid = mdf[mdf['SEASON'] == '2022-23']
test  = mdf[mdf['SEASON'] >= '2023-24']

m_lgb = lgb.LGBMRegressor(
    n_estimators=3000, learning_rate=0.02, num_leaves=31, min_child_samples=30,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.6, reg_lambda=1.0,
    random_state=42, n_jobs=-1)
m_lgb.fit(train[feats], train['pct_cap'],
          eval_set=[(valid[feats], valid['pct_cap'])],
          callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)])

pred = m_lgb.predict(test[feats])
mae  = mean_absolute_error(test['pct_cap'], pred)
print(f"test R2: {r2_score(test['pct_cap'], pred):.3f}  MAE: {mae:.4f}  (~${mae*SALARY_CAP['2025-26']:,.0f})")

imp = pd.Series(m_lgb.booster_.feature_importance(importance_type='gain'),
                index=feats).sort_values(ascending=False)
print(f"\n>0 gain: {(imp>0).sum()}/{len(feats)}")
print(imp.head(25))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.035423 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 146633
[LightGBM] [Info] Number of data points in the train set: 2673, number of used features: 1044
[LightGBM] [Info] Start training from score 0.079533
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[494]	valid_0's l2: 0.00121168
test R2: 0.840  MAE: 0.0247  (~$3,819,475)

>0 gain: 991/1044
EXPERIENCE                    71.577133
MIN_rs                        41.886930
MIN_rs_lag1                   23.610403
FGM_rs                        23.410902
FRONT_CT_TOUCHES_rs           15.025712
FGM_rs_lag1                   12.346203
LBR_WAR_prior1                 9.663248
MIN_rs_lag2                    8.207685
FTM_rs                         7.575481
MAX_PCT_ELIGIBLE               6.613066
FGM_rs_lag2                    6.516227
DRAFT_POSITION           

# SHAP Feature Importance

In [81]:
explainer   = shap.TreeExplainer(m_lgb)
X_all       = mdf[feats]
shap_values = explainer.shap_values(X_all)   # (n_players, n_features) matrix of SHAP values              # (n_rows, n_feats)

# --- global: mean |SHAP| — the honest importance ranking ---
mean_abs = (pd.Series(np.abs(shap_values).mean(0), index=feats)
          .sort_values(ascending=False))
print("top 50 by mean |SHAP|:"); print(mean_abs.head(50))

top 50 by mean |SHAP|:
EXPERIENCE                         0.025102
MIN_rs                             0.009121
MIN_rs_lag1                        0.005396
FRONT_CT_TOUCHES_rs                0.004644
FGM_rs                             0.004455
DRAFT_POSITION                     0.004317
MIN_rs_lag2                        0.003895
MAX_PCT_ELIGIBLE                   0.002826
LBR_WAR_prior1                     0.002344
LBR_WAR_prior2                     0.001940
FGM_rs_lag1                        0.001918
CLOSESTDEF_4_6_FGM_rs_lag2         0.001898
FRONT_CT_TOUCHES_rs_lag1           0.001690
FGM_rs_lag2                        0.001436
FRONT_CT_TOUCHES_rs_lag2           0.001315
FTM_rs                             0.001203
PIE_rs_lag1                        0.001121
OPP_PTS_OFF_TOV_rs                 0.001071
TOTAL_MIN_rs_lag2                  0.001055
LBR_LEBRON_prior1                  0.000976
ALLNBA_WT_EVER_lag1                0.000908
FTM_rs_lag2                        0.000821
LBR_LEBRO

In [105]:
# reuse the explainer + shap values from the global cell (don't recompute)
# sv, expl, Xall, feats, mdf all already exist

def explain_player(name, season, top_n=30):
    row = mdf[(mdf['PLAYER_NAME'] == name) & (mdf['SEASON'] == season)]
    if row.empty:
        print(f"no row for {name} {season}"); return
    i   = mdf.index.get_loc(row.index[0])
    cap = SALARY_CAP[season]

    contrib = pd.Series(shap_values[i], index=feats)
    top = contrib.reindex(contrib.abs().sort_values(ascending=False).index).head(top_n)

    base = expl.expected_value
    pred = row['pred_pct_cap'].iloc[0]; actual = row['pct_cap'].iloc[0]
    print(f"\n{name} — {season}")
    print(f"  market baseline:  {base:6.1%}  (${base*cap:,.0f})")
    print(f"  model prediction: {pred:6.1%}  (${pred*cap:,.0f})")
    print(f"  actual salary:    {actual:6.1%}  (${actual*cap:,.0f})")
    print(f"  → {'OVER' if actual>pred else 'UNDER'}paid vs market by ${abs(actual-pred)*cap:,.0f}\n")
    print("  what pushed the market price (+ up / − down):")
    for f, v in top.items():
        print(f"    {'+' if v>0 else '−'} {f:32s} {v*cap:>+12,.0f}")

# the players you just flagged
explain_player("jaylen brown", "2025-26")
explain_player("christian braun", "2025-26")
# explain_player("julius randle", "2025-26")
# explain_player("jonathan kuminga", "2025-26")
# explain_player("tyler herro", "2025-26")
explain_player("ziaire williams", "2025-26")


jaylen brown — 2025-26
  market baseline:    8.0%  ($12,306,789)
  model prediction:  30.3%  ($46,890,763)
  actual salary:     34.4%  ($53,142,264)
  → OVERpaid vs market by $6,251,501

  what pushed the market price (+ up / − down):
    + EXPERIENCE                         +7,811,051
    + MIN_rs                             +4,675,321
    + FGM_rs                             +2,481,327
    + CLOSESTDEF_4_6_FGM_rs_lag2         +1,761,890
    + FRONT_CT_TOUCHES_rs                +1,759,623
    + MIN_rs_lag1                        +1,496,844
    + DRAFT_POSITION                     +1,400,920
    + FGM_rs_lag1                          +935,360
    + FTM_rs                               +802,196
    + MIN_rs_lag2                          +799,838
    + LBR_WAR_prior1                       +799,415
    + MAX_PCT_ELIGIBLE                     +775,653
    + ALLNBA_WT_EVER_lag1                  +736,713
    + LBR_WAR_prior2                       +714,098
    + LBR_LEBRON_prior1             


Alex Caruso 2025-26
  actual 11.7% | predicted 6.8% | resid $7,632,640
    LBR_D_LEBRON_prior1: 1.906
    LBR_O_LEBRON_prior1: -0.122
    LBR_LEBRON_prior1: 1.784

Luguentz Dort 2025-26
  actual 11.8% | predicted 5.5% | resid $9,696,690
    LBR_D_LEBRON_prior1: 1.186
    LBR_O_LEBRON_prior1: -0.426
    LBR_LEBRON_prior1: 0.76

Jrue Holiday 2025-26
  actual 21.0% | predicted 18.1% | resid $4,353,743
    LBR_D_LEBRON_prior1: 0.697
    LBR_O_LEBRON_prior1: -0.333
    LBR_LEBRON_prior1: 0.364

Jaden Mcdaniels 2025-26
  actual 15.8% | predicted 11.7% | resid $6,290,483
    LBR_D_LEBRON_prior1: 1.506
    LBR_O_LEBRON_prior1: 0.0
    LBR_LEBRON_prior1: 1.505

Herbert Jones 2025-26
  actual 9.0% | predicted 11.4% | resid $-3,721,106
    LBR_D_LEBRON_prior1: 0.994
    LBR_O_LEBRON_prior1: -1.89
    LBR_LEBRON_prior1: -0.896

Dorian Finney-Smith 2025-26
  actual 8.2% | predicted 4.5% | resid $5,664,817
    LBR_D_LEBRON_prior1: -0.275
    LBR_O_LEBRON_prior1: -0.834
    LBR_LEBRON_prior1: -1.108

# Predictions / Results

## Over/Under Paid

In [15]:
mdf = mdf.copy()
mdf['pred_pct_cap'] = m_lgb.predict(Xall)
mdf['residual']     = mdf['pct_cap'] - mdf['pred_pct_cap']   # +overpaid  /  −underpaid
mdf['resid_usd']    = mdf['residual'] * mdf['SEASON'].map(SALARY_CAP)

In [ ]:
#most over/under paid since 15-16

cols = ['PLAYER_NAME','SEASON','pct_cap','pred_pct_cap','resid_usd']
print("\n💸 most OVERPAID (model says they earn above their profile):")
print(mdf.sort_values('residual', ascending=False)[cols].head(20).to_string(index=False))
print("\n💎 most UNDERPAID:")
print(mdf.sort_values('residual')[cols].head(20).to_string(index=False))


💸 most OVERPAID (model says they earn above their profile):
      PLAYER_NAME  SEASON  pct_cap  pred_pct_cap    resid_usd
        john wall 2022-23 0.330490      0.093480 2.930753e+07
      ben simmons 2024-25 0.279228      0.077270 2.839287e+07
      ben simmons 2022-23 0.286674      0.112598 2.152542e+07
    fred vanvleet 2023-24 0.300000      0.159658 1.908943e+07
    fred vanvleet 2024-25 0.304767      0.168693 1.913038e+07
russell westbrook 2022-23 0.374391      0.246739 1.578479e+07
   jonathan isaac 2024-25 0.177825      0.055355 1.721768e+07
     myles turner 2022-23 0.283608      0.162708 1.494981e+07
       kevin love 2022-23 0.221931      0.105525 1.439408e+07
  khris middleton 2025-26 0.215305      0.103339 1.731529e+07
       og anunoby 2024-25 0.260605      0.148845 1.571207e+07
       og anunoby 2025-26 0.255866      0.145630 1.704771e+07
   gordon hayward 2023-24 0.244958      0.136450 1.475938e+07
      jalen suggs 2025-26 0.226322      0.118225 1.671680e+07
  khris m

In [ ]:
#most over/under paid since 2025-26

this_yr = mdf[mdf['SEASON'] == '2025-26'].copy()
print("most overpaid 2025-26:")
print(this_yr.sort_values('residual', ascending=False)[cols].head(60).to_string(index=False))
print("\nmost underpaid 2025-26:")
print(this_yr.sort_values('residual')[cols].head(60).to_string(index=False))

most overpaid 2025-26:
          PLAYER_NAME  SEASON  pct_cap  pred_pct_cap    resid_usd
      khris middleton 2025-26 0.215305      0.103339 1.731529e+07
           og anunoby 2025-26 0.255866      0.145630 1.704771e+07
          jalen suggs 2025-26 0.226322      0.118225 1.671680e+07
          zach lavine 2025-26 0.307149      0.203044 1.609948e+07
         jordan poole 2025-26 0.205941      0.102497 1.599743e+07
giannis antetokounmpo 2025-26 0.350000      0.256665 1.443397e+07
        anthony davis 2025-26 0.350000      0.258117 1.420947e+07
          evan mobley 2025-26 0.300000      0.214672 1.319569e+07
   karl-anthony towns 2025-26 0.343636      0.261276 1.273677e+07
      zion williamson 2025-26 0.255072      0.173445 1.262340e+07
       darius garland 2025-26 0.255072      0.184436 1.092362e+07
          paul george 2025-26 0.334090      0.265437 1.061712e+07
         jimmy butler 2025-26 0.350000      0.283363 1.030522e+07
             naz reid 2025-26 0.139361      0.073820 

In [113]:
feat_idx = {f: i for i, f in enumerate(feats)}
mdf_reset = mdf.reset_index(drop=True)

# define cohorts by residual (this season, or all — your call)
# resid_usd > 0 = underpaid (actual > predicted)... wait, check YOUR sign convention!
# earlier: resid = actual - predicted, positive = OVERpaid. confirm before trusting direction.
under_mask = (mdf_reset['resid_usd'] < mdf_reset['resid_usd'].quantile(0.10)).values  # most underpaid decile
base_mask  = np.ones(len(mdf_reset), dtype=bool)  # everyone as baseline

# mean SHAP per feature for cohort vs baseline, then the DIFFERENCE (enrichment)
under_mean = sv[under_mask].mean(axis=0)
base_mean  = sv[base_mask].mean(axis=0)
enrichment = under_mean - base_mean   # features pushing underpaid group's price UP more than typical

enr = pd.Series(enrichment, index=feats).sort_values(ascending=False)
print("=== Features driving the UNDERPAID cohort's price UP, vs baseline ===")
print(enr.head(20))
print("\n=== ...pushing DOWN more than typical ===")
print(enr.tail(10))

=== Features driving the UNDERPAID cohort's price UP, vs baseline ===
EXPERIENCE                    0.010513
MIN_rs                        0.002395
MIN_rs_lag2                   0.002283
MIN_rs_lag1                   0.001931
FGM_rs                        0.001458
CLOSESTDEF_4_6_FGM_rs_lag2    0.001226
LBR_WAR_prior2                0.001094
MAX_PCT_ELIGIBLE              0.000981
LBR_WAR_prior1                0.000964
FGM_rs_lag2                   0.000873
FRONT_CT_TOUCHES_rs           0.000776
FGM_rs_lag1                   0.000774
FRONT_CT_TOUCHES_rs_lag2      0.000548
TOTAL_MIN_rs_lag2             0.000537
RM_POSS_PCT_rs                0.000420
FRONT_CT_TOUCHES_rs_lag1      0.000367
FTM_rs                        0.000297
OPP_PTS_OFF_TOV_rs            0.000237
AST_TO_PASS_PCT_rs            0.000230
FTM_rs_lag2                   0.000214
dtype: float64

=== ...pushing DOWN more than typical ===
USG_PCT_rs_lag2                    -0.000076
PAINT_TOUCH_AST_PCT_rs             -0.000077
PC

## LEBRON vs Box Score/Tracking Stats Evaluation

In [111]:
# assuming you have both models' predictions. If model-1 preds are in app_meta or you can regenerate:
m1 = pd.read_parquet("data/app_meta_v2.parquet")[['PLAYER_NAME','SEASON','pred_pct_cap']].rename(columns={'pred_pct_cap':'pred_boxonly'})
# model-2 preds = your current mdf['pred_pct_cap']
m2 = mdf[['PLAYER_NAME', 'SEASON','pred_pct_cap']].rename(columns={'pred_pct_cap':'pred_lebron'})

cmp = m1.merge(m2, on=['PLAYER_NAME','SEASON'])
cmp['lebron_effect'] = cmp['pred_lebron'] - cmp['pred_boxonly']   # how much LEBRON moved the price
S = "2025-26"
c = cmp[cmp.SEASON==S].copy()

print("=== LEBRON RAISED their price most (box score UNDERrated their impact) ===")
print(c.nlargest(50,'lebron_effect')[['PLAYER_NAME','pred_boxonly','pred_lebron','lebron_effect']].to_string(index=False))
print("\n=== LEBRON LOWERED their price most (box score OVERrated their impact) ===")
print(c.nsmallest(50,'lebron_effect')[['PLAYER_NAME','pred_boxonly','pred_lebron','lebron_effect']].to_string(index=False))

=== LEBRON RAISED their price most (box score UNDERrated their impact) ===
          PLAYER_NAME  pred_boxonly  pred_lebron  lebron_effect
         myles turner      0.138886     0.169965       0.031079
          lamelo ball      0.193328     0.223755       0.030427
giannis antetokounmpo      0.229976     0.256665       0.026689
        derrick white      0.176167     0.200604       0.024437
           al horford      0.105192     0.128410       0.023219
     payton pritchard      0.137303     0.159551       0.022249
   brandin podziemski      0.045139     0.066038       0.020899
         caris levert      0.088617     0.109060       0.020443
         franz wagner      0.200296     0.220439       0.020144
          alex caruso      0.047987     0.067698       0.019712
           malik monk      0.112797     0.130650       0.017852
        demar derozan      0.229817     0.247643       0.017826
        stephen curry      0.316370     0.334136       0.017765
          brook lopez      0.

In [39]:
# 1. get a clean PLAYER_NAME+SEASON -> PLAYER_ID mapping from mdf (which HAS the id)
id_map = (mdf.reset_index()[['PLAYER_NAME','SEASON','PLAYER_ID']]
             .drop_duplicates(subset=['PLAYER_NAME','SEASON']))
id_map['PLAYER_ID'] = id_map['PLAYER_ID'].astype(str).str.strip()

# 2. attach PLAYER_ID to your comparison frame c
c2 = c.merge(id_map, on=['PLAYER_NAME','SEASON'], how='left')
print("rows missing PLAYER_ID after id attach:", c2['PLAYER_ID'].isna().sum())  # should be ~0

# 3. pull raw CURRENT-season role from your lebron source (for descriptive analysis)
role_raw = lebron[['PLAYER_ID','SEASON','LBR_OffRole','LBR_DefRole']].copy()
role_raw['PLAYER_ID'] = role_raw['PLAYER_ID'].astype(str).str.strip()

# 4. join role on PLAYER_ID + SEASON
c_with_role = c2.merge(role_raw, on=['PLAYER_ID','SEASON'], how='left')
print("rows missing role:", c_with_role['LBR_OffRole'].isna().sum())

# 5. the money tables
print("\n=== LEBRON effect by OFFENSIVE role ===")
print(c_with_role.groupby('LBR_OffRole')['lebron_effect'].agg(['mean','count']).sort_values('mean'))
print("\n=== LEBRON effect by DEFENSIVE role ===")
print(c_with_role.groupby('LBR_DefRole')['lebron_effect'].agg(['mean','count']).sort_values('mean'))

rows missing PLAYER_ID after id attach: 0
rows missing role: 0

=== LEBRON effect by OFFENSIVE role ===
                            mean  count
LBR_OffRole                            
Post Scorer            -0.002581      2
Movement Shooter       -0.002259     80
Athletic Finisher      -0.001749     10
Roll + Cut Big         -0.001374     34
Off Screen Shooter     -0.000655     12
Stationary Shooter     -0.000587     50
Stretch Big            -0.000414     21
Shot Creator           -0.000071     89
Secondary Ball Handler  0.000584     22
Primary Ball Handler    0.001668     51
Slasher                 0.001856      3
Versatile Big           0.007645      7

=== LEBRON effect by DEFENSIVE role ===
                     mean  count
LBR_DefRole                     
Wing Stopper    -0.001631     53
Point of Attack -0.000999     74
Helper          -0.000875     91
Mobile Big      -0.000280     59
Anchor Big       0.000100     15
Chaser           0.000672     71
Low Activity     0.003451     1

In [40]:
# does the LEBRON effect split cleanly by offensive vs defensive impact?
lbr_split = lebron[['PLAYER_ID','SEASON','LBR_O_LEBRON','LBR_D_LEBRON']].copy()
lbr_split['PLAYER_ID'] = lbr_split['PLAYER_ID'].astype(str).str.strip()
cs = c_with_role.merge(lbr_split, on=['PLAYER_ID','SEASON'], how='left')

# players whose impact is defense-tilted vs offense-tilted
cs['def_tilt'] = cs['LBR_D_LEBRON'] - cs['LBR_O_LEBRON']   # positive = defense-first player
cs['tilt_bucket'] = pd.qcut(cs['def_tilt'], 4, labels=['Offense-first','Off-lean','Def-lean','Defense-first'])
print(cs.groupby('tilt_bucket')['lebron_effect'].agg(['mean','count']))

                   mean  count
tilt_bucket                   
Offense-first  0.001185     96
Off-lean      -0.001701     95
Def-lean      -0.000839     95
Defense-first -0.000187     95


/var/folders/gd/xcr4zw7j3ml77xcpdgf__f2c0000gn/T/ipykernel_57722/3127558265.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(cs.groupby('tilt_bucket')['lebron_effect'].agg(['mean','count']))


In [108]:
# pick each player's MOST RECENT season (veteran, paid, has prior-year LEBRON)
t = mdf.reset_index()
defenders = ['alex caruso','luguentz dort','jrue holiday','jaden mcdaniels',
             'herbert jones','dorian finney-smith','derrick white','ausar thompson']
for name in defenders:
    rows = t[t['PLAYER_NAME'].str.lower() == name]
    if rows.empty:
        continue
    r = rows.sort_values('SEASON').iloc[-1]         # <-- latest season, not first
    print(f"\n{name.title()} {r['SEASON']}")
    print(f"  actual {r['pct_cap']:.1%} | predicted {r['pred_pct_cap']:.1%} | resid ${r['resid_usd']:,.0f}")
    for c in ['LBR_D_LEBRON_prior1','LBR_O_LEBRON_prior1','LBR_LEBRON_prior1']:
        if c in t.columns: print(f"    {c}: {r[c]}")


Alex Caruso 2025-26
  actual 11.7% | predicted 6.8% | resid $7,632,640
    LBR_D_LEBRON_prior1: 1.906
    LBR_O_LEBRON_prior1: -0.122
    LBR_LEBRON_prior1: 1.784

Luguentz Dort 2025-26
  actual 11.8% | predicted 5.5% | resid $9,696,690
    LBR_D_LEBRON_prior1: 1.186
    LBR_O_LEBRON_prior1: -0.426
    LBR_LEBRON_prior1: 0.76

Jrue Holiday 2025-26
  actual 21.0% | predicted 18.1% | resid $4,353,743
    LBR_D_LEBRON_prior1: 0.697
    LBR_O_LEBRON_prior1: -0.333
    LBR_LEBRON_prior1: 0.364

Jaden Mcdaniels 2025-26
  actual 15.8% | predicted 11.7% | resid $6,290,483
    LBR_D_LEBRON_prior1: 1.506
    LBR_O_LEBRON_prior1: 0.0
    LBR_LEBRON_prior1: 1.505

Herbert Jones 2025-26
  actual 9.0% | predicted 11.4% | resid $-3,721,106
    LBR_D_LEBRON_prior1: 0.994
    LBR_O_LEBRON_prior1: -1.89
    LBR_LEBRON_prior1: -0.896

Dorian Finney-Smith 2025-26
  actual 8.2% | predicted 4.5% | resid $5,664,817
    LBR_D_LEBRON_prior1: -0.275
    LBR_O_LEBRON_prior1: -0.834
    LBR_LEBRON_prior1: -1.108

In [112]:
# map feature name -> column index in the sv matrix
feat_idx = {f: i for i, f in enumerate(feats)}
lbr_feats = [f for f in feats if 'LBR_' in f]   # the LEBRON features

# you need the raised/lowered player groups from your c (comparison) frame.
# get the row positions in mdf for those players (2025-26)
raised_names  = c.nlargest(30, 'lebron_effect')['PLAYER_NAME'].tolist()
lowered_names = c.nsmallest(30, 'lebron_effect')['PLAYER_NAME'].tolist()

# boolean masks over mdf rows (match on name + season if c is single-season)
mdf_reset = mdf.reset_index(drop=True)   # ensure positional index aligns with sv rows
raised_mask  = mdf_reset['PLAYER_NAME'].isin(raised_names).values
lowered_mask = mdf_reset['PLAYER_NAME'].isin(lowered_names).values

# average SHAP (in model units) for each LEBRON feature, per cohort
def lbr_profile(mask, label):
    print(f"\n=== {label}: mean SHAP per LEBRON feature ===")
    for f in lbr_feats:
        col = feat_idx[f]
        print(f"  {f:22s} {sv[mask, col].mean():+.5f}")

lbr_profile(raised_mask,  "LEBRON-RAISED players")
lbr_profile(lowered_mask, "LEBRON-LOWERED players")


=== LEBRON-RAISED players: mean SHAP per LEBRON feature ===
  LBR_LEBRON_prior1      +0.00079
  LBR_O_LEBRON_prior1    +0.00029
  LBR_D_LEBRON_prior1    +0.00004
  LBR_WAR_prior1         +0.00252
  LBR_LEBRON_prior2      +0.00059
  LBR_O_LEBRON_prior2    +0.00015
  LBR_D_LEBRON_prior2    +0.00002
  LBR_WAR_prior2         +0.00185

=== LEBRON-LOWERED players: mean SHAP per LEBRON feature ===
  LBR_LEBRON_prior1      -0.00055
  LBR_O_LEBRON_prior1    -0.00024
  LBR_D_LEBRON_prior1    -0.00018
  LBR_WAR_prior1         +0.00001
  LBR_LEBRON_prior2      -0.00050
  LBR_O_LEBRON_prior2    -0.00012
  LBR_D_LEBRON_prior2    -0.00016
  LBR_WAR_prior2         -0.00008


## Salary Rankings

In [56]:
ranked = (mdf[mdf['SEASON']=='2025-26']
          .assign(deserved_usd = mdf['pred_pct_cap']*SALARY_CAP['2025-26'])
          .sort_values('pred_pct_cap', ascending=False)
          [['PLAYER_NAME','pred_pct_cap','deserved_usd','pct_cap','resid_usd']])
print(ranked.head(100).to_string(index=False))

            PLAYER_NAME  pred_pct_cap  deserved_usd  pct_cap     resid_usd
          stephen curry      0.334136  5.167310e+07 0.385438  7.933716e+06
          kawhi leonard      0.325459  5.033124e+07 0.323317 -3.312353e+05
           lebron james      0.320642  4.958637e+07 0.340305  3.040781e+06
           nikola jokic      0.319402  4.939457e+07 0.357101  5.829953e+06
            luka doncic      0.319378  4.939087e+07 0.297449 -3.391214e+06
           devin booker      0.315774  4.883350e+07 0.343636  4.308765e+06
       donovan mitchell      0.310441  4.800881e+07 0.300000 -1.614709e+06
           kevin durant      0.309975  4.793666e+07 0.353764  6.771945e+06
           james harden      0.309819  4.791263e+07 0.253369 -8.729937e+06
           jaylen brown      0.303212  4.689076e+07 0.343636  6.251501e+06
          pascal siakam      0.298123  4.610383e+07 0.294545 -5.533131e+05
            joel embiid      0.295082  4.563350e+07 0.357101  9.591023e+06
shai gilgeous-alexander  

## Impact of Years of Experience 

In [115]:
# the 14+ tier's most over-predicted players (most negative resid = model thinks worth >> paid)
old = mdf[mdf['EXPERIENCE'] >= 14].copy()
worst = old.sort_values('resid_usd').head(10)   # most negative = most over-predicted
print("=== Most over-predicted 14+ vets (model says worth WAY more than paid) ===")
print(worst[['PLAYER_NAME','SEASON','EXPERIENCE','pct_cap','pred_pct_cap','resid_usd']]
      .assign(resid_M=lambda x: (x.resid_usd/1e6).round(1))
      .drop(columns='resid_usd').to_string(index=False))

=== Most over-predicted 14+ vets (model says worth WAY more than paid) ===
      PLAYER_NAME  SEASON  EXPERIENCE  pct_cap  pred_pct_cap  resid_M
russell westbrook 2025-26        17.0 0.014848      0.137130    -18.9
    demar derozan 2024-25        15.0 0.166444      0.276711    -15.5
russell westbrook 2024-25        16.0 0.023500      0.129506    -14.9
       al horford 2025-26        18.0 0.036761      0.128410    -14.2
       kyle lowry 2024-25        18.0 0.014848      0.113412    -13.9
    demar derozan 2025-26        16.0 0.158878      0.247643    -13.7
russell westbrook 2023-24        15.0 0.028200      0.128600    -13.7
       chris paul 2024-25        19.0 0.074402      0.168048    -13.2
     james harden 2024-25        15.0 0.239379      0.327327    -12.4
    demar derozan 2023-24        14.0 0.210262      0.294509    -11.5


In [116]:
# B, step 1: is there a systematic over/under-prediction pattern by experience?
import numpy as np
d = mdf.copy()
d['exp_tier'] = pd.cut(d['EXPERIENCE'],
                       bins=[-1, 2, 5, 9, 13, 100],
                       labels=['0-2 (rookie)','3-5 (young)','6-9 (prime)','10-13 (vet)','14+ (old)'])
# residual = actual - predicted; POSITIVE = model UNDER-predicts (player paid MORE than model says)
# NEGATIVE = model OVER-predicts (model thinks worth more than paid)
summary = d.groupby('exp_tier').agg(
    n=('resid_usd','size'),
    mean_resid_M=('resid_usd', lambda x: x.mean()/1e6),
    median_resid_M=('resid_usd', lambda x: x.median()/1e6),
    mean_pred_pct=('pred_pct_cap','mean'),
    mean_actual_pct=('pct_cap','mean'),
).round(3)
print(summary.to_string())

                 n  mean_resid_M  median_resid_M  mean_pred_pct  mean_actual_pct
exp_tier                                                                        
0-2 (rookie)  1118        -0.024          -0.015          0.029            0.029
3-5 (young)    963         0.026          -0.112          0.084            0.084
6-9 (prime)    880        -0.144          -0.146          0.132            0.131
10-13 (vet)    487         0.155          -0.094          0.140            0.141
14+ (old)      192        -0.532          -0.148          0.133            0.130


/var/folders/gd/xcr4zw7j3ml77xcpdgf__f2c0000gn/T/ipykernel_57722/288107395.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary = d.groupby('exp_tier').agg(


In [60]:
# what does the model predict for Paolo's profile?
paolo = mdf[(mdf.PLAYER_NAME.str.contains('banchero', case=False))]
print(paolo[['SEASON','pct_cap','pred_pct_cap']].to_string())

       SEASON   pct_cap  pred_pct_cap
2055  2022-23  0.089403      0.079354
2056  2023-24  0.085340      0.074778
2057  2024-25  0.086500      0.109628
2058  2025-26  0.099160      0.116296


In [74]:
rookies = mdf[mdf['EXPERIENCE'].isin([0, 1, 2, 3])].copy()

# predict all at once — dtypes preserved, no object coercion
rookies['pred_now'] = m_lgb.predict(rookies[feats])

# bumped version: copy, set experience to 4, predict all at once
bumped = rookies.copy()
bumped['EXPERIENCE'] = 4.0
rookies['pred_postrookie'] = m_lgb.predict(bumped[feats])

rookies['jump'] = rookies['pred_postrookie'] - rookies['pred_now']

print(rookies[rookies.SEASON=='2025-26']
      .sort_values('pred_now', ascending=False)
      [['PLAYER_NAME','EXPERIENCE','pred_now','pred_postrookie','jump']]
      .head(20).to_string(index=False))


       PLAYER_NAME  EXPERIENCE  pred_now  pred_postrookie     jump
 victor wembanyama         2.0  0.121223         0.226532 0.105309
    paolo banchero         3.0  0.116296         0.242225 0.125929
    brandon miller         2.0  0.096712         0.199146 0.102434
     keegan murray         3.0  0.095890         0.173683 0.077793
     chet holmgren         3.0  0.095548         0.157378 0.061830
      vj edgecombe         0.0  0.082435         0.176024 0.093589
     amen thompson         2.0  0.082352         0.188635 0.106283
      cooper flagg         0.0  0.081195         0.202043 0.120848
    jalen williams         3.0  0.079350         0.159729 0.080380
       jalen duren         3.0  0.070643         0.145531 0.074889
    stephon castle         1.0  0.070280         0.139220 0.068940
      jabari smith         3.0  0.066837         0.136328 0.069491
brandin podziemski         2.0  0.066038         0.118715 0.052677
    ausar thompson         2.0  0.065538         0.111300 0.04

In [76]:
rookies = mdf[mdf['EXPERIENCE'].isin([4, 5, 6, 7])].copy()

# predict all at once — dtypes preserved, no object coercion
rookies['pred_now'] = m_lgb.predict(rookies[feats])

# bumped version: copy, set experience to 4, predict all at once
bumped = rookies.copy()
bumped['EXPERIENCE'] = 8
rookies['pred_postrookie'] = m_lgb.predict(bumped[feats])

rookies['jump'] = rookies['pred_postrookie'] - rookies['pred_now']

print(rookies[rookies.SEASON=='2025-26']
      .sort_values('pred_now', ascending=False)
      [['PLAYER_NAME','EXPERIENCE','pred_now','pred_postrookie','jump']]
      .head(20).to_string(index=False))

            PLAYER_NAME  EXPERIENCE  pred_now  pred_postrookie     jump
            luka doncic         7.0  0.319378         0.344126 0.024748
shai gilgeous-alexander         7.0  0.290413         0.310419 0.020006
          jalen brunson         7.0  0.282640         0.303022 0.020382
        anthony edwards         5.0  0.270627         0.288059 0.017432
         scottie barnes         4.0  0.246877         0.260345 0.013468
           tyrese maxey         5.0  0.240093         0.257531 0.017438
        cade cunningham         4.0  0.236110         0.255318 0.019208
         alperen sengun         4.0  0.227867         0.245511 0.017644
            lamelo ball         5.0  0.223755         0.235212 0.011457
           franz wagner         4.0  0.220439         0.234006 0.013567
              ja morant         6.0  0.218311         0.223074 0.004763
         michael porter         7.0  0.216493         0.221837 0.005344
            evan mobley         4.0  0.214672         0.225346 0

In [71]:
# take Wemby's most recent feature row, bump EXPERIENCE, re-predict
wemby_row = mdf[(mdf.PLAYER_NAME.str.contains('wembanyama', case=False)) & (mdf.SEASON=='2025-26')].copy()
print("current EXPERIENCE:", wemby_row['EXPERIENCE'].values)
print("current pred:", wemby_row['pred_pct_cap'].values)

# simulate him at 5 years experience (post-rookie)
wemby_sim = wemby_row.copy()
wemby_sim['EXPERIENCE'] = 4   # or whatever his year-5 value would be
# also bump MAX_PCT_ELIGIBLE if he crosses an eligibility threshold!
pred_sim = m_lgb.predict(wemby_sim[feats])
print("predicted pct_cap at 5 yrs experience:", pred_sim)

current EXPERIENCE: [2.]
current pred: [0.12122296]
predicted pct_cap at 5 yrs experience: [0.22653232]


In [ ]:
# take Paolo's most recent feature row, bump EXPERIENCE, re-predict
paolo_row = mdf[(mdf.PLAYER_NAME.str.contains('banchero', case=False)) & (mdf.SEASON=='2025-26')].copy()
print("current EXPERIENCE:", paolo_row['EXPERIENCE'].values)
print("current pred:", paolo_row['pred_pct_cap'].values)

# simulate him at 5 years experience (post-rookie)
paolo_sim = paolo_row.copy()
paolo_sim['EXPERIENCE'] = 5   # or whatever his year-5 value would be
# also bump MAX_PCT_ELIGIBLE if he cr  osses an eligibility threshold!
pred_sim = m_lgb.predict(paolo_sim[feats])
print("predicted pct_cap at 5 yrs experience:", pred_sim)

current EXPERIENCE: [3.]
current pred: [0.11629621]
predicted pct_cap at 5 yrs experience: [0.2422254]


In [66]:
for exp in [2, 3, 4, 5, 6, 8, 10, 12]:
    row = paolo_row.copy()
    row['EXPERIENCE'] = float(exp)
    print(f"EXP={exp}: pred={m_lgb.predict(row[feats])[0]:.4f}")

EXP=2: pred=0.1163
EXP=3: pred=0.1163
EXP=4: pred=0.2422
EXP=5: pred=0.2422
EXP=6: pred=0.2426
EXP=8: pred=0.2555
EXP=10: pred=0.2602
EXP=12: pred=0.2641


## All-NBA Decay Check Between V1 & V2

In [28]:
# Load OLD predictions and compare directly for faded stars
old = pd.read_parquet("data/app_meta_v1.parquet")
if "PLAYER_NAME" not in old.columns: old = old.reset_index()

faded = ["russell westbrook","demar derozan","paul george","rudy gobert",
         "pascal siakam","ja morant"]
S = "2025-26"

o = (old[old["SEASON"]==S][["PLAYER_NAME","pred_pct_cap"]]
       .rename(columns={"pred_pct_cap":"pred_OLD"}))
n = mdf[mdf["SEASON"]==S][["PLAYER_NAME","pred_pct_cap"]].rename(columns={"pred_pct_cap":"pred_NEW"})
c = o.merge(n, on="PLAYER_NAME").query("PLAYER_NAME.str.lower() in @faded", engine="python")
c["change_pct_cap"] = c["pred_NEW"] - c["pred_OLD"]
print(c.sort_values("change_pct_cap").to_string(index=False))

      PLAYER_NAME  pred_OLD  pred_NEW  change_pct_cap
      paul george  0.284414  0.266745       -0.017670
russell westbrook  0.150191  0.142372       -0.007819
      rudy gobert  0.225275  0.219233       -0.006042
    demar derozan  0.229862  0.229817       -0.000045
    pascal siakam  0.281280  0.287385        0.006106
        ja morant  0.179120  0.201718        0.022599


In [42]:
# --- pick a recent season to inspect ---
SEASON = "2025-26"   # change if you want a different one

# --- 1. find faded stars: strong career All-NBA résumé, nothing recent ---
m = mdf[mdf["SEASON"] == SEASON].copy()

# sanity: confirm the columns exist / see what All-NBA cols you actually have
allnba_cols = [c for c in m.columns if "ALLNBA" in c.upper()]
print("All-NBA columns present:", allnba_cols)
print()

faded = m[(m["ALLNBA_WT_EVER"] > 0) & (m["ALLNBA_WT3"] == 0)] \
          .sort_values("ALLNBA_WT_EVER", ascending=False)

print(f"=== Faded stars in {SEASON} (high career All-NBA, 0 in last 3 yrs) ===")
print(faded[["PLAYER_NAME", "ALLNBA_WT_EVER", "ALLNBA_WT3"]].head(15).to_string(index=False))
print()

# --- 2. how much is ALLNBA_WT_EVER paying these guys in SHAP $? ---
faded_names = faded["PLAYER_NAME"].head(15).tolist()
sh = shap_long[(shap_long["SEASON"] == SEASON) &
          (shap_long["PLAYER_NAME"].isin(faded_names)) &
          (shap_long["feature"] == "ALLNBA_WT_EVER")]

print("=== ALLNBA_WT_EVER SHAP $ contribution for those faded stars ===")
print(sh[["PLAYER_NAME", "shap_usd"]]
        .sort_values("shap_usd", ascending=False)
        .to_string(index=False))
print()

# --- 3. contrast: current stars (recent All-NBA) — the boost SHOULD fire here ---
current = m[m["ALLNBA_WT3"] > 0].sort_values("ALLNBA_WT3", ascending=False)
cur_names = current["PLAYER_NAME"].head(10).tolist()
sh_cur = shap_long[(shap_long["SEASON"] == SEASON) &
              (shap_long["PLAYER_NAME"].isin(cur_names)) &
              (shap_long["feature"] == "ALLNBA_WT_EVER")]
print("=== ALLNBA_WT_EVER SHAP $ for CURRENT stars (should be legitimately high) ===")
print(sh_cur[["PLAYER_NAME", "shap_usd"]]
        .sort_values("shap_usd", ascending=False)
        .to_string(index=False))

All-NBA columns present: ['ALLNBA_PRIOR3', 'ALLNBA_PRIOR_EVER', 'ALLNBA_WT_EVER', 'ALLNBA_WT3', 'ALLNBA_WT_EVER_lag1', 'ALLNBA_WT_EVER_lag2']

=== Faded stars in 2025-26 (high career All-NBA, 0 in last 3 yrs) ===
      PLAYER_NAME  ALLNBA_WT_EVER  ALLNBA_WT3
    demar derozan        0.908123           0
russell westbrook        0.905883           0
      paul george        0.794183           0
      rudy gobert        0.698936           0
    pascal siakam        0.672363           0
        ja morant        0.632812           0
   draymond green        0.187712           0
   andre drummond        0.056314           0
    klay thompson        0.056314           0

=== ALLNBA_WT_EVER SHAP $ contribution for those faded stars ===
Empty DataFrame
Columns: [PLAYER_NAME, shap_usd]
Index: []

=== ALLNBA_WT_EVER SHAP $ for CURRENT stars (should be legitimately high) ===
Empty DataFrame
Columns: [PLAYER_NAME, shap_usd]
Index: []


# Save Model, Results, SHAP

In [ ]:
# shap = pd.read_parquet("data/shap_v1.parquet")

In [48]:
N_STORE = 25
rows = []
cap_by_season = mdf['SEASON'].map(SALARY_CAP).values

for r in range(len(mdf)):
    cap = cap_by_season[r]
    player_sv = sv[r, :]
    top_idx = np.argsort(np.abs(player_sv))[::-1][:N_STORE]   # THIS player's top 25
    for j in top_idx:
        f = feats[j]
        rows.append((mdf['PLAYER_NAME'].iloc[r], mdf['SEASON'].iloc[r],
                     f, player_sv[j], player_sv[j] * cap, mdf[f].iloc[r]))

shap_long = pd.DataFrame(rows, columns=['PLAYER_NAME','SEASON','feature','shap','shap_usd','feat_value'])
shap_long.to_parquet("data/shap_v2.parquet")   # overwrite — same version, corrected generation
print("rows:", len(shap_long), "| distinct features stored:", shap_long['feature'].nunique())

# app_meta unchanged — regenerate only if you want, it's identical

rows: 105700 | distinct features stored: 346


In [49]:
sd = pd.read_parquet("data/shap_v2.parquet")
print("distinct features in local shap_v2:", sd["feature"].nunique())
print("total rows:", len(sd))

distinct features in local shap_v2: 346
total rows: 105700


In [ ]:
for f in ["data/app_meta_v2.parquet", "data/shap_v2.parquet"]:
    if os.path.exists(f):
        df = pd.read_parquet(f)
        print(f"{f}: EXISTS, {df.shape}, cols={list(df.columns)[:4]}")
    else:
        print(f"{f}: MISSING")

data/app_meta_v2.parquet: EXISTS, (4228, 6), cols=['PLAYER_NAME', 'SEASON', 'pct_cap', 'pred_pct_cap']
data/shap_v2.parquet: EXISTS, (105700, 6), cols=['PLAYER_NAME', 'SEASON', 'feature', 'shap']


In [126]:
# clean model (idempotent guard for any future lag re-runs)
model = model.loc[:, ~model.columns.duplicated()]

# 1) the trained model — both formats
m_lgb.booster_.save_model("data/salary_model_v1.txt")
joblib.dump(m_lgb, "data/salary_model_v1.pkl")

# 2) the exact feature list + order (model is useless without this)
json.dump(feats, open("data/feats_v1.json", "w"))

# 3) the residual table — the product, queryable
out_cols = ['PLAYER_NAME','SEASON','pct_cap','pred_pct_cap','resid_usd']
mdf[out_cols].to_parquet("data/residuals_v1.parquet")

# 4) the scoreable population (everyone, not just training rows) for the app
mdf.to_parquet("data/mdf_v1.parquet")

print("saved:", report)   # log the prefilter summary so you know what built this
print(f"R2 0.847 | MAE $3.75M | {len(feats)} features | {len(mdf)} rows")

saved: {'start': 1487, 'const': 1, 'empty': 2, 'collinear': 449, 'keep': 1035}
R2 0.847 | MAE $3.75M | 1037 features | 4228 rows


In [104]:
# ============================================================
# SAVE v2_lebron ARTIFACTS  (LEBRON model — SEPARATE from box-only v2)
# Does NOT overwrite or renumber any box-only files.
# ============================================================

SUFFIX = "v2_lebron"
N_STORE = 25

# ---- sanity: confirm this sv/m_lgb/mdf/feats are the LEBRON ones ----
assert any('LBR_' in f for f in feats), "feats has no LEBRON features — wrong model loaded!"
print(f"Saving LEBRON artifacts | {len(feats)} feats (incl. {sum('LBR_' in f for f in feats)} LEBRON) | {len(mdf)} rows")

# ---- 1) trained model (both formats) + exact feature order ----
m_lgb.booster_.save_model(f"data/salary_model_{SUFFIX}.txt")
joblib.dump(m_lgb, f"data/salary_model_{SUFFIX}.pkl")
json.dump(feats, open(f"data/feats_{SUFFIX}.json", "w"))

# ---- 2) per-player SHAP (NOT global), LEBRON features INCLUDED ----
rows = []
cap_by_season = mdf['SEASON'].map(SALARY_CAP).values
for r in range(len(mdf)):
    cap = cap_by_season[r]
    player_sv = sv[r, :]
    top_idx = np.argsort(np.abs(player_sv))[::-1][:N_STORE]   # THIS player's top 25
    for j in top_idx:
        f = feats[j]
        rows.append((mdf['PLAYER_NAME'].iloc[r], mdf['SEASON'].iloc[r],
                     f, player_sv[j], player_sv[j] * cap, mdf[f].iloc[r]))
shap_long = pd.DataFrame(rows, columns=['PLAYER_NAME','SEASON','feature',
                                        'shap','shap_usd','feat_value'])
shap_long.to_parquet(f"data/shap_{SUFFIX}.parquet")

# ---- 3) predictions / meta (waterfall anchors) ----
base = float(expl.expected_value)
meta = mdf[['PLAYER_NAME','SEASON','pct_cap','pred_pct_cap','resid_usd']].copy()
meta['baseline_pct'] = base
meta.to_parquet(f"data/app_meta_{SUFFIX}.parquet")

# ---- 4) scoreable population ----
mdf.to_parquet(f"data/mdf_{SUFFIX}.parquet")

print(f"\n✅ saved {SUFFIX}:")
print(f"   shap rows: {len(shap_long):,} | distinct feats: {shap_long['feature'].nunique()}")
print(f"   baseline pct_cap: {round(base,4)}")
print(f"   R2 0.840 | MAE ~$3.82M | {len(feats)} feats | {len(mdf)} rows")

Saving LEBRON artifacts | 1044 feats (incl. 8 LEBRON) | 4228 rows

✅ saved v2_lebron:
   shap rows: 105,700 | distinct feats: 342
   baseline pct_cap: 0.0796
   R2 0.840 | MAE ~$3.82M | 1044 feats | 4228 rows


# Streamlit App

In [ ]:
# # global ranking → the features worth storing for the app
# glob = pd.Series(np.abs(sv).mean(0), index=feats).sort_values(ascending=False)
# DISPLAY_FEATS = glob.head(25).index.tolist()          # the app shows top contributors per player
# display_idx   = [feats.index(f) for f in DISPLAY_FEATS]

# # build a long-format SHAP table: one row per (player-season, feature) = easy for plotly
# rows = []
# cap_by_season = mdf['SEASON'].map(SALARY_CAP).values
# for r in range(len(mdf)):
#     cap = cap_by_season[r]
#     for f, j in zip(DISPLAY_FEATS, display_idx):
#         rows.append((mdf['PLAYER_NAME'].iloc[r], mdf['SEASON'].iloc[r],
#                      f, sv[r, j], sv[r, j] * cap, mdf[f].iloc[r]))

# shap_long = pd.DataFrame(rows, columns=['PLAYER_NAME','SEASON','feature',
#                                         'shap','shap_usd','feat_value'])
# shap_long.to_parquet("data/shap_v1.parquet")

# # also store the per-player baseline + prediction so the waterfall has its anchors
# base = float(expl.expected_value)
# meta = mdf[['PLAYER_NAME','SEASON','pct_cap','pred_pct_cap','resid_usd']].copy()
# meta['baseline_pct'] = base
# meta.to_parquet("data/app_meta_v1.parquet")

# print(f"stored SHAP for {len(DISPLAY_FEATS)} features × {len(mdf)} player-seasons "
#       f"= {len(shap_long):,} rows ({shap_long.memory_usage(deep=True).sum()/1e6:.1f} MB)")
# print("baseline pct_cap:", round(base, 4))

stored SHAP for 25 features × 4228 player-seasons = 105,700 rows (24.2 MB)
baseline pct_cap: 0.0796


In [103]:
existing = pd.read_parquet("data/shap_v2.parquet")
print("distinct features in existing shap_v2:", existing['feature'].nunique())

distinct features in existing shap_v2: 346


In [129]:
pip install -r data/requirements.txt

94527.50s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 37.0 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.2/731.2 kB 13.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 49.1 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 41.7 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: packaging0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/20 [protobuf]
    Found existing installation: packaging 26.2━━━━━━━━━━━━━━━  4/20 [protobuf]
    Uninstalling packaging-26.2:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/20 [protobuf]
      Successfully uninstalled packaging-26.2━━━━━━━━━━━━━━━━━  4/20 [protobuf]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20/20 [streamlit]20 [streamlit]]
Note: you may need to restart the kernel to use updated packages.
